# 04 — ReAct, Reflexion and Program-of-Thought

Status of three agentic patterns. None of them has a frozen benchmark result. Labels used in every notebook: **MEASURED** (a real provider or real run produced the number), **ESTIMATED** (derived here from measured data by a stated formula), **SIMULATED** (mock runner, no real model), **NOT_RUN**, **UNKNOWN** (the evidence does not say). Implementation and wiring are reported separately from evaluation: code that exists is not a measured capability.

In [1]:
import json
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 11  # the pre-registered bootstrap seed; reused for every derived computation here
random.seed(SEED)
np.random.seed(SEED)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "EVIDENCE_MAP.md").exists())  # research/council-ablation
EV = ROOT / "evidence"
JEV = ROOT.parent / "jev" / "evidence"  # the JEV study's canonical copy in this repository
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)


def load(rel):
    return json.loads((EV / rel).read_text())


def jsonl(rel):
    return [json.loads(x) for x in (EV / rel).read_text().splitlines() if x.strip()]


def final_calls(rel):
    """The harness cache is append-only: a re-asked call (Amendments 1 and 2) appends a new row and the
    last row per (member, case) is the one analysed. Earlier rows stay in the file as recorded calls."""
    return pd.DataFrame(jsonl(rel)).drop_duplicates(["member", "case_id"], keep="last").reset_index(drop=True)


R = load("council/results.json")
REG = {r["id"]: r for r in load("reasoning_methods/reasoning_methods_registry.json")}
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)
print("evidence files:", sorted(str(p.relative_to(ROOT)) for p in EV.rglob("*") if p.is_file()))


python 3.12.12 | pandas 3.1.0 | numpy 2.5.4 | seed 11
evidence files: ['evidence/SOURCE_MANIFEST.json', 'evidence/council/calibration_by_member.csv', 'evidence/council/cases.csv', 'evidence/council/cases.json', 'evidence/council/conditions.csv', 'evidence/council/cost_latency_by_condition.csv', 'evidence/council/judge_calls.csv', 'evidence/council/judge_calls.jsonl', 'evidence/council/member_calls.csv', 'evidence/council/member_calls.jsonl', 'evidence/council/member_calls_budget300_archive.csv', 'evidence/council/member_calls_budget300_archive.jsonl', 'evidence/council/results.json', 'evidence/decision_methods/ablation-2026-10-20.json', 'evidence/notebook_runs.json', 'evidence/publication_scan.json', 'evidence/reasoning_methods/method_status_summary.csv', 'evidence/reasoning_methods/reasoning_methods_registry.json', 'evidence/reasoning_methods/reasoning_strategies_mock_benchmark.md', 'evidence/reasoning_methods/research_plans.json', 'evidence/replay_check.json']


In [2]:
#: Evaluation state per method, with the evidence that sets it. Explicit, not inferred from text.
EVALUATION = {
    "deterministic-baseline": "MEASURED",      # reference arm of the decision-methods ablation
    "single-strong-model": "MEASURED",         # condition A of the council benchmark
    "council": "MEASURED",                     # conditions B1-D, F (harness, not the product endpoint)
    "proposer-judge": "MEASURED",              # as condition E (judge over C) in the harness only
    "critic-veto": "NOT_RUN",
    "critic-selective-council": "NOT_RUN",
    "cot": "SIMULATED (mock runner)",
    "tot": "SIMULATED (mock runner)",
    "got": "SIMULATED (mock runner)",
    "self-consistency": "NOT_RUN",
    "react": "NOT_RUN",
    "reflexion": "NOT_RUN",
    "pot-evidence": "NOT_RUN",
    "mcts": "NOT_RUN",
    "jev": "MEASURED",                         # real provider calls on decision-corpus-v1
    "agentic-llm-decision": "MEASURED",        # decision-methods ablation, small local model
    "hermes-agent-patterns": "NOT_RUN",
}


def status_row(method_id):
    """Implementation / wiring / evaluation labels, derived from the registry snapshot only."""
    r = REG[method_id]
    implemented = "NOT_IMPLEMENTED" if r["wiring"] == "NOT_PRESENT" else "IMPLEMENTED"
    wired = "WIRED" if r["wiring"] in {"WIRED", "DECISION_AFFECTING", "OBSERVED", "EVALUATED"} else "NOT_WIRED"
    evaluation = EVALUATION[method_id]
    return {"method": r["name"], "implemented": implemented, "wired": wired, "authority": r["authority"],
            "evaluation": evaluation, "production": r["production"], "recommendation": r["recommendation"],
            "result": r["result"], "limitations": r["limitations"]}


In [3]:
pd.DataFrame([status_row(m) for m in ("react", "reflexion", "pot-evidence")]).T

,0,1,2
method,ReAct (reason + act tool loop),Reflexion (self-critique after failure),Program-of-Thought / evidence enforcement
implemented,NOT_IMPLEMENTED,IMPLEMENTED,IMPLEMENTED
wired,NOT_WIRED,WIRED,WIRED
authority,NONE,ADVISORY,NONE
evaluation,NOT_RUN,NOT_RUN,NOT_RUN
production,NOT_IN_PRODUCT,API_ONLY,IN_PRODUCT
recommendation,RESEARCH,RESEARCH,ADOPT
result,UNKNOWN,UNKNOWN,UNKNOWN
limitations,NL orchestration is a deterministic intent planner (athena.nl_orchestration.planner); no tool loop.,"One reflection step, not iterative Reflexion; never evaluated.","Partial: Insights, Recommends, Executive/Engineer Summary and comparison/causal narratives are not checked number-by-number against evid..."


In [4]:
for m in ("react", "reflexion", "pot-evidence"):
    r = REG[m]
    print(f"{r['name']}\n  definition: {r['definition']}\n  runtime: {r['runtime']}\n  next step: {r['next_step']}\n")

ReAct (reason + act tool loop)
  definition: Interleaved reasoning and tool calls until an answer.
  runtime: —
  next step: Notebook 04.

Reflexion (self-critique after failure)
  definition: After a failed attempt the model diagnoses and proposes a retry.
  runtime: athena.flow.crews.auto_retry (prompt auto_retry.diagnose_and_propose)
  next step: Notebook 04.

Program-of-Thought / evidence enforcement
  definition: Numbers come from canonical computation; the model only explains them.
  runtime: athena.storytelling.validator.validate_report; athena.reporting.grounding.ensure_grounded; athena.validation.evidence
  next step: P0 4B: make canonical evidence the mandatory source for every narrative number.



**RESULT.**

* **ReAct** — `NOT_IMPLEMENTED`. Natural-language orchestration in Athena is a deterministic intent planner; there is no reason-act tool loop. `NOT_RUN`.
* **Reflexion** — one diagnose-and-retry step is wired (advisory, API only). It is not iterative Reflexion and has never been evaluated: `NOT_RUN`.
* **Program-of-Thought / evidence enforcement** — wired: numbers in validated reports come from canonical computation, and the model only explains them. Coverage is partial (some narrative surfaces are not checked number by number). No ablation measures its effect: `NOT_RUN`.

No accuracy, cost or latency figure is claimed for any of the three.